<a href="https://colab.research.google.com/github/alejocast2511/LENGUAJE-DE-PROGRAMACION-DCS/blob/main/ESCENARIOS_DE_INVERSION_DESCUBRIENDO_TU_PERFIL_INVERSOR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
"""
Proyecto: Análisis de portafolios por Simulación Monte Carlo

"""
!pip install yfinance streamlit pyngrok scipy pandas numpy matplotlib
# %%writefile app.py

# ==== 1 Importar librerías ====

# Librerías estándar
import sys
import subprocess
import threading
import time
import io
import math
from typing import List

# Librerías de análisis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Librerías financieras y visuales
import yfinance as yf
import streamlit as st
from pyngrok import ngrok

In [32]:
# ==== (2) Definición de carteras actualizadas ===

CARTERAS = {
    'Conservadora':{
        'tickers': ['KO', 'PG', 'JNJ', 'TLT', 'GLD'],  #   coca-cola, procter & gamble, jhonson & jhonson, ishares 20+ year treasury bond, spdr gold shares.
        'pesos': [0.25, 0.25, 0.20, 0.20, 0.10]
    },
    'Balanceada': {
        'tickers': ['AAPL', 'MSFT', 'VTI', 'VNQ', 'DBC'],  # apple inc, microsoft corp, vanguard total stock etf, vanguard real estate etf, invesco db commodity index trankin found
        'pesos': [0.20, 0.20, 0.30, 0.15, 0.15]
    },
    'Arriesgada': {
        'tickers': ['TSLA', 'NVDA', 'META', 'BTC-USD', 'SPY'],  # Tesla inc, nvidia copr, meta plataforms, bitcoin, spdr s&p 500 etf trust
        'pesos': [0.25, 0.25, 0.20, 0.15, 0.15]
    }
}

In [33]:
# ==== (3) Funciones utilitarias ====

DIAS_ANIO = 252

def generar_precios_sinteticos(tickers: List[str], periodo: str = '5y') -> pd.DataFrame:
    """Genera precios sintéticos por ticker usando un GBM simple."""
    dias_map = {'1y': 252, '3y': 252*3, '5y': 252*5, '10y': 252*10, 'max': 252*5}
    dias = dias_map.get(periodo, 252*5)
    fechas = pd.bdate_range(end=pd.Timestamp.today(), periods=dias)
    rng = np.random.default_rng(seed=42)
    df = pd.DataFrame(index=fechas)
    for tk in tickers:
        S0 = 100.0 * (1 + (abs(hash(tk)) % 50) / 100)  # precio inicial pseudoaleatorio
        mu = 0.07    # retorno esperado anual (aleatorio razonable)
        sigma = 0.15 # volatilidad anual (aleatorio razonable)
        dt = 1/DIAS_ANIO
        increments = rng.normal(loc=(mu*dt), scale=(sigma*math.sqrt(dt)), size=dias)
        logS = np.log(S0) + np.cumsum(increments)
        df[tk] = np.exp(logS)
    return df

In [34]:
def descargar_datos(tickers: List[str], periodo: str = '5y') -> pd.DataFrame:
    """Descarga precios ajustados con yfinance si está disponible; si no, usa precios sintéticos."""
    # Normalizar input a lista
    if isinstance(tickers, str):
        tickers = [tickers]

    mode_sintetico = False

    if mode_sintetico or yf is None:
        print("Modo sintético: generando series de precios sintéticos.")
        return generar_precios_sinteticos(tickers, periodo=periodo)

    try:
        data = yf.download(tickers, period=periodo, interval='1d', progress=False)
        if isinstance(data, pd.DataFrame) and 'Adj Close' in data.columns:
            adj = data['Adj Close']
        elif isinstance(data, pd.DataFrame) and data.columns.nlevels > 1:
            adj = data.xs('Adj Close', axis=1, level=1, drop_level=True)
        else:
            adj = data

        if isinstance(adj, pd.Series):
            adj = adj.to_frame(name=adj.name)

        adj = adj.dropna(how='all')
        if adj.empty:
            print("Advertencia: yfinance devolvió DataFrame vacío; usando datos sintéticos.")
            return generar_precios_sinteticos(tickers, periodo=periodo)
        return adj
    except Exception as e:
        print(f"Error descargando con yfinance: {e}; se usarán datos sintéticos.")
        return generar_precios_sinteticos(tickers, periodo=periodo)


In [35]:
def calcular_retornos_log(precios: pd.DataFrame) -> pd.DataFrame:
    precios = precios.sort_index()
    precios = precios.dropna(axis=1, how='all')
    return np.log(precios / precios.shift(1)).dropna()

In [36]:
def simulacion_montecarlo(retornos: pd.DataFrame, pesos: np.ndarray, dias_horizonte: int = 252,
                          n_simul: int = 10000, pt_inicial: float = 150000.0, random_seed: int = None):
    if random_seed is not None:
        np.random.seed(random_seed)

    mu = retornos.mean().values * DIAS_ANIO
    sigma = retornos.cov().values * DIAS_ANIO

    # Regularización para evitar errores en Cholesky
    eps = 1e-8
    try:
        chol = np.linalg.cholesky(sigma + eps * np.eye(sigma.shape[0]))
    except np.linalg.LinAlgError:
        chol = np.linalg.cholesky(sigma + 1e-6 * np.eye(sigma.shape[0]))

    n_assets = len(mu)
    resultados_final = np.zeros(n_simul)
    trayectorias = np.zeros((n_simul, dias_horizonte))

    dt = 1/DIAS_ANIO
    for i in range(n_simul):
        z = np.random.normal(size=(dias_horizonte, n_assets))
        shocks = z.dot(chol.T)
        drift = (mu / DIAS_ANIO) - 0.5 * np.diag(sigma) / DIAS_ANIO
        factors = np.exp(drift[np.newaxis, :] + shocks * math.sqrt(dt))
        precios_relativos = np.cumprod(factors, axis=0)
        valores = (precios_relativos * pesos[np.newaxis, :]).sum(axis=1) * pt_inicial
        resultados_final[i] = valores[-1]
        trayectorias[i, :] = valores

    return {
        'final_values': resultados_final,
        'trayectorias': trayectorias,
        'mean_final': float(np.mean(resultados_final)),
        'median_final': float(np.median(resultados_final)),
        'std_final': float(np.std(resultados_final)),
        'percentiles': np.percentile(resultados_final, [5,25,50,75,95]).tolist()
    }

In [37]:
# ==== (4) Cuestionario y perfil ====

def determinar_perfil(respuestas: dict) -> str:
    horizonte = respuestas.get('horizonte', 5)
    aversion = respuestas.get('aversion_perdida', 3)
    objetivo = respuestas.get('objetivo', 'crecimiento')
    tolerancia = respuestas.get('tolerancia_vol', 'media')

    score = 0
    if horizonte >= 10:
        score += 2
    elif horizonte >= 5:
        score += 1
    score += (3 - aversion)
    if objetivo == 'preservacion':
        score -= 2
    if tolerancia == 'baja':
        score -= 1
    elif tolerancia == 'alta':
        score += 1

    if score <= -1:
        return 'Conservadora'
    elif score <= 2:
        return 'Balanceada'
    else:
        return 'Arriesgada'


In [38]:
# ==== (5) Graficas asociadas ====

def graficar_histograma(final_values: np.ndarray):
    fig, ax = plt.subplots()
    ax.hist(final_values, bins=50)
    ax.set_title('Distribución del valor final del portafolio')
    ax.set_xlabel('Valor final (USD)')
    ax.set_ylabel('Frecuencia')
    return fig

def graficar_trayectorias(trayectorias: np.ndarray, n_plot: int = 50):
    fig, ax = plt.subplots()
    for i in range(min(n_plot, trayectorias.shape[0])):
        ax.plot(trayectorias[i, :], alpha=0.4)
    ax.set_title(f'Trayectorias simuladas ({min(n_plot, trayectorias.shape[0])} ejemplos)')
    ax.set_xlabel('Días')
    ax.set_ylabel('Valor del portafolio (USD)')
    return fig



In [39]:
# ==============================================================
# SECCIÓN 4: SIMULACIÓN MONTE CARLO + INTERFAZ STREAMLIT
# ==============================================================

# En esta sección se combinan todas las funciones anteriores para crear
# una aplicación interactiva que identifica el perfil del inversionista,
# selecciona la cartera adecuada y simula su comportamiento esperado.

import streamlit as st
from pyngrok import ngrok
import matplotlib.pyplot as plt

# --------------------------------------------------------------
# 1️⃣ CONFIGURACIÓN DE STREAMLIT
# --------------------------------------------------------------

def configurar_streamlit():
    """
    Configura la página de Streamlit y su apariencia general.
    """
    st.set_page_config(
        page_title="Simulación Monte Carlo de Portafolios de Inversión",
        page_icon="💹",
        layout="wide"
    )

    st.title("💼 Simulación de Portafolios de Inversión con Monte Carlo")
    st.markdown("""
    Este simulador permite identificar el perfil del inversionista
    (conservador, balanceado o arriesgado) y evaluar el rendimiento
    potencial de su portafolio a través de **Simulación Monte Carlo**.
    """)

# --------------------------------------------------------------
# 2️⃣ CUESTIONARIO DE PERFIL DEL INVERSOR
# --------------------------------------------------------------

def cuestionario_perfil():
    """
    Muestra preguntas al usuario para determinar su perfil de riesgo.
    """
    st.header("🧭 Cuestionario del Inversionista")

    riesgo = st.selectbox(
        "¿Qué nivel de riesgo estás dispuesto a asumir?",
        ["Bajo", "Moderado", "Alto"]
    )

    horizonte = st.selectbox(
        "¿Cuál es tu horizonte de inversión?",
        ["Corto plazo (1-2 años)", "Mediano plazo (3-5 años)", "Largo plazo (más de 5 años)"]
    )

    st.write("Tus respuestas serán usadas para definir el portafolio más adecuado.")
    return riesgo, horizonte

# --------------------------------------------------------------
# 3️⃣ DETERMINAR PERFIL DE INVERSOR
# --------------------------------------------------------------

def determinar_perfil(riesgo, horizonte):
    """
    Devuelve el tipo de cartera según las respuestas del usuario.
    """
    if riesgo == "Bajo" and "Corto" in horizonte:
        return "Conservadora"
    elif riesgo == "Moderado" or "Mediano" in horizonte:
        return "Balanceada"
    else:
        return "Arriesgada"

# --------------------------------------------------------------
# 4️⃣ MOSTRAR RESULTADOS DE LA SIMULACIÓN
# --------------------------------------------------------------

def mostrar_resultados(simulaciones, perfil, cartera):
    """
    Presenta los resultados de la simulación Monte Carlo.
    """
    st.header(f"📊 Resultados de la Cartera {perfil}")

    st.write("**Activos en cartera:**")
    for t, p in zip(cartera['tickers'], cartera['pesos']):
        st.write(f"- {t}: {p*100:.1f}%")

    # Mostrar métricas principales
    st.subheader("Indicadores Estadísticos")
    st.write(f"**Rendimiento esperado anual:** {simulaciones['rendimiento_esperado']*100:.2f}%")
    st.write(f"**Riesgo estimado (desviación estándar):** {simulaciones['riesgo']*100:.2f}%")
    st.write(f"**Escenario optimista (percentil 95):** {simulaciones['percentil_95']*100:.2f}%")
    st.write(f"**Escenario pesimista (percentil 5):** {simulaciones['percentil_5']*100:.2f}%")

    # Histograma
    fig, ax = plt.subplots(figsize=(8,4))
    ax.hist(simulaciones['resultados'], bins=40, color='skyblue', edgecolor='black')
    ax.set_title("Distribución de rendimientos simulados")
    ax.set_xlabel("Rendimiento Simulado (%)")
    ax.set_ylabel("Frecuencia")
    st.pyplot(fig)

# --------------------------------------------------------------
# 5️⃣ FUNCIÓN PRINCIPAL DE EJECUCIÓN
# --------------------------------------------------------------

def main():
    configurar_streamlit()
    riesgo, horizonte = cuestionario_perfil()

    if st.button("Generar Simulación"):
        perfil = determinar_perfil(riesgo, horizonte)
        cartera = CARTERAS[perfil]

        st.success(f"Perfil identificado: **{perfil}**")
        st.info("Descargando datos y ejecutando simulaciones...")

        datos = obtener_datos(cartera['tickers'])
        rendimientos = rendimientos_log(datos)
        retornos_medios, covarianza = calcular_metricas(rendimientos)
        simulaciones = simulacion_montecarlo(rendimientos, cartera['pesos'])

        mostrar_resultados(simulaciones, perfil, cartera)

# --------------------------------------------------------------
# 6️⃣ EJECUTAR STREAMLIT CON PYNGROK
# --------------------------------------------------------------

# Este bloque permite ejecutar Streamlit dentro de Google Colab.
# Crea un túnel público hacia el puerto 8501 para ver la app en el navegador.

if __name__ == "__main__":
    # Abrir túnel
    public_url = ngrok.connect(8501)
    print("🌐 URL pública de la aplicación:", public_url)
    # Ejecutar Streamlit
    !streamlit run app.py &>/dev/null&

🌐 URL pública de la aplicación: NgrokTunnel: "https://pseudomonastically-commonsensible-janey.ngrok-free.dev" -> "http://localhost:8501"


In [40]:
!pip install yfinance streamlit pyngrok scipy pandas numpy matplotlib
from pyngrok import ngrok

# Inicia ngrok
ngrok.kill()
public_url = ngrok.connect(8501)
print("🌐 URL pública:", public_url)

# Lanza streamlit en segundo plano
!streamlit run app.py --server.port 8501 & sleep 5

🌐 URL pública: NgrokTunnel: "https://pseudomonastically-commonsensible-janey.ngrok-free.dev" -> "http://localhost:8501"


2025-10-24 23:37:17.956 Port 8501 is already in use
